# WSMTE — 06_evaluation.ipynb
**LOCAL PC** | Day 5 Tasks 5.13–5.25

All metrics, statistical tests, and figure generation.

**Prerequisites:** All files downloaded from Kaggle sessions:
- `ablation/ablation_results_AG.csv`
- `ablation/ablation_results_H.csv`
- `results/saved_models/config_h_best.keras`
- `results/figures/shap_summary.png` (already generated by notebook 05)
- `data/processed/X_test.npy`, `y_clf_test.npy`, `y_reg_test.npy`
- `data/processed/final_dataset.csv`

In [ ]:
# ── Cell 1: Imports + paths ────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for local PC
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json
from scipy import stats

from config.config import CONFIG

# Ensure output directories exist
for d in [CONFIG['figures_dir'], CONFIG['tables_dir'], CONFIG['models_dir']]:
    os.makedirs(d, exist_ok=True)

print('Imports OK')
print(f"Figures dir: {CONFIG['figures_dir']}")
print(f"Tables dir : {CONFIG['tables_dir']}")

## Step 1 — Merge Ablation Results (A–H)

In [ ]:
# ── Cell 2: Load + merge AG and H results ─────────────────────────────────
df_ag = pd.read_csv('ablation/ablation_results_AG.csv')
df_h  = pd.read_csv('ablation/ablation_results_H.csv')

print(f'Results A-G shape : {df_ag.shape}  configs: {df_ag["config"].unique().tolist()}')
print(f'Results H shape   : {df_h.shape}  configs: {df_h["config"].unique().tolist()}')

ablation = pd.concat([df_ag, df_h], ignore_index=True)
ablation.to_csv(CONFIG['ablation_results'], index=False)
print(f'\nMerged -> {CONFIG["ablation_results"]}  shape: {ablation.shape}')
print(ablation.groupby('config')['accuracy'].count().rename('n_runs').to_string())

## Step 2 — Per-Config Summary Statistics

In [ ]:
# ── Cell 3: Mean / Max / Std per config ───────────────────────────────────
summary_cols = ['accuracy', 'balanced_accuracy', 'auc', 'precision', 'recall', 'f1']
reg_cols = ['rmse', 'mae', 'r2']

summary = ablation.groupby('config').agg(
    mean_acc=('accuracy', 'mean'),
    max_acc=('accuracy', 'max'),
    std_acc=('accuracy', 'std'),
    mean_bacc=('balanced_accuracy', 'mean'),
    mean_auc=('auc', 'mean'),
    mean_f1=('f1', 'mean'),
    mean_rmse=('rmse', 'mean'),
    n_runs=('accuracy', 'count'),
).round(4)
print(summary.to_string())

summary.to_csv(CONFIG['tables_dir'] + 'ablation_summary.csv')
print(f"\nSaved -> {CONFIG['tables_dir']}ablation_summary.csv")

## Step 3 — Wilcoxon Signed-Rank Test: Config B vs Config H

In [ ]:
# ── Cell 4: Wilcoxon test ──────────────────────────────────────────────────
acc_b = ablation[ablation['config'] == 'B']['accuracy'].values
acc_h = ablation[ablation['config'] == 'H']['accuracy'].values

# Use min(len) for paired test (B=10, H=30 -> compare first 10 pairs)
n_pairs = min(len(acc_b), len(acc_h))
stat, p_value = stats.wilcoxon(acc_b[:n_pairs], acc_h[:n_pairs])

print(f'Wilcoxon signed-rank test: Config B vs Config H')
print(f'  n pairs     : {n_pairs}')
print(f'  Config B    : mean={acc_b.mean():.4f}  std={acc_b.std():.4f}')
print(f'  Config H    : mean={acc_h.mean():.4f}  std={acc_h.std():.4f}')
print(f'  W statistic : {stat:.4f}')
print(f'  p-value     : {p_value:.6f}')
print(f'  Significant : {"YES" if p_value < 0.05 else "NO"} (alpha=0.05)')

wilcoxon_result = {
    'test': 'Wilcoxon signed-rank',
    'config_a': 'B', 'config_b': 'H',
    'n_pairs': n_pairs,
    'mean_B': round(float(acc_b.mean()), 4),
    'mean_H': round(float(acc_h.mean()), 4),
    'W_statistic': round(float(stat), 4),
    'p_value': round(float(p_value), 6),
    'significant_at_0_05': bool(p_value < 0.05),
}
with open(CONFIG['results_dir'] + 'wilcoxon_result.json', 'w') as f:
    json.dump(wilcoxon_result, f, indent=2)
print(f"Saved -> {CONFIG['results_dir']}wilcoxon_result.json")

## Step 4 — Ablation Comparison Bar Chart

In [ ]:
# ── Cell 5: ablation_comparison.png ───────────────────────────────────────
config_order = ['A', 'B', 'C', 'D', 'E', 'G', 'H']
means = [summary.loc[c, 'mean_acc'] if c in summary.index else 0 for c in config_order]
stds  = [summary.loc[c, 'std_acc']  if c in summary.index else 0 for c in config_order]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(config_order, means, yerr=stds, capsize=5,
              color=['#4C72B0','#55A868','#C44E52','#8172B2',
                     '#CCB974','#64B5CD','#E8735A'],
              alpha=0.85, edgecolor='black', linewidth=0.6)

# Kotekar baseline
ax.axhline(0.5853, color='red', linestyle='--', linewidth=1.2,
           label='Kotekar baseline (0.5853)')

for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{m:.4f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Ablation Config')
ax.set_ylabel('Mean Test Accuracy')
ax.set_title('WSMTE Ablation Study — Mean Accuracy per Config (±1 std)')
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] + 'ablation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {CONFIG['figures_dir']}ablation_comparison.png")

## Step 5 — Load Best Config H Model + Test Predictions

In [ ]:
# ── Cell 6: Load model + test data ────────────────────────────────────────
import tensorflow as tf

X_test      = np.load(CONFIG['processed_data_dir'] + 'X_test.npy')
y_clf_test  = np.load(CONFIG['processed_data_dir'] + 'y_clf_test.npy')
y_reg_test  = np.load(CONFIG['processed_data_dir'] + 'y_reg_test.npy')

print(f'X_test shape     : {X_test.shape}')
print(f'y_clf_test shape : {y_clf_test.shape}  labels={set(y_clf_test.tolist())}')
print(f'y_reg_test shape : {y_reg_test.shape}')

h_model = tf.keras.models.load_model(
    CONFIG['models_dir'] + 'config_h_best.keras'
)
print(f'Config H model loaded OK. Trainable params: {h_model.count_params():,}')

raw_preds = h_model(X_test, training=False)
y_reg_pred  = raw_preds[0].numpy().ravel()
y_clf_proba = raw_preds[1].numpy().ravel()
y_clf_pred  = (y_clf_proba >= 0.5).astype(int)

print(f'clf_proba range : {y_clf_proba.min():.4f} to {y_clf_proba.max():.4f}')
print(f'reg_pred  range : {y_reg_pred.min():.6f} to {y_reg_pred.max():.6f}')

## Step 6 — Confusion Matrix

In [ ]:
# ── Cell 7: confusion_matrix.png ──────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_clf_test, y_clf_pred)
print('Confusion Matrix (rows=actual, cols=predicted):')
print(f'  TN={cm[0,0]}  FP={cm[0,1]}')
print(f'  FN={cm[1,0]}  TP={cm[1,1]}')

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Down (0)', 'Up (1)']
)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Config H — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] + 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {CONFIG['figures_dir']}confusion_matrix.png")

## Step 7 — AUC-ROC Curve

In [ ]:
# ── Cell 8: auc_roc_curve.png ─────────────────────────────────────────────
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_clf_test, y_clf_proba)
auc_score = roc_auc_score(y_clf_test, y_clf_proba)
print(f'AUC-ROC: {auc_score:.4f}')

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, lw=2, color='#4C72B0', label=f'Config H (AUC = {auc_score:.4f})')
ax.plot([0,1],[0,1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Config H (Test Set)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] + 'auc_roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {CONFIG['figures_dir']}auc_roc_curve.png")

## Step 8 — Training Loss Curves
> Uses per-run history from the best Config H run (reconstructed from ablation CSV).

In [ ]:
# ── Cell 9: loss_curves.png ───────────────────────────────────────────────
# Re-train best Config H run to capture history (or load if saved separately)
# Here we do a quick 1-run retrain with the best seed to get clean curves.
from src.models.wsmte import build_wsmte, build_wsmte_pso
from src.training.callbacks import get_callbacks

# Load best PSO weights
with open(CONFIG['results_dir'] + '../pso_weights.json') as f:
    pso_info = json.load(f)
pso_w = np.array([pso_info['w_lstm'], pso_info['w_tcn'], pso_info['w_gru']])
print(f'PSO weights: LSTM={pso_w[0]:.4f}, TCN={pso_w[1]:.4f}, GRU={pso_w[2]:.4f}')

# Load best seed from results
best_seed = int(ablation[ablation['config']=='H']['accuracy'].idxmax())
best_seed_val = int(ablation[ablation['config']=='H'].iloc[best_seed]['seed'])
tf.random.set_seed(best_seed_val)
np.random.seed(best_seed_val)

X_train = np.load(CONFIG['processed_data_dir'] + 'X_train.npy')
X_val   = np.load(CONFIG['processed_data_dir'] + 'X_val.npy')
y_reg_train = np.load(CONFIG['processed_data_dir'] + 'y_reg_train.npy')
y_clf_train = np.load(CONFIG['processed_data_dir'] + 'y_clf_train.npy')
y_reg_val   = np.load(CONFIG['processed_data_dir'] + 'y_reg_val.npy')
y_clf_val   = np.load(CONFIG['processed_data_dir'] + 'y_clf_val.npy')

lc_model = build_wsmte_pso(CONFIG, pso_w)
lc_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['learning_rate'])
)
history = lc_model.fit(
    X_train, [y_reg_train, y_clf_train],
    validation_data=(X_val, [y_reg_val, y_clf_val]),
    epochs=CONFIG['pso_finetune_epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=get_callbacks(CONFIG, run_id='loss_curve_run'),
    verbose=0,
)
print(f'Loss curve training done. Epochs run: {len(history.history["loss"])}')

In [ ]:
# ── Cell 10: Plot loss curves ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
epochs = range(1, len(history.history['loss']) + 1)
ax.plot(epochs, history.history['loss'],     label='Train loss', lw=2)
ax.plot(epochs, history.history['val_loss'], label='Val loss',   lw=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MTL uncertainty-weighted)')
ax.set_title('Config H — Training vs Validation Loss')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] + 'loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {CONFIG['figures_dir']}loss_curves.png")

## Step 9 — Granger Causality Tests

In [ ]:
# ── Cell 11: Granger causality (company + market polarity, lags 1-5) ───────
from src.evaluation.granger_test import run_granger_tests

# Load final_dataset.csv — has raw Close column and sentiment columns
final_df = pd.read_csv(CONFIG['processed_data_dir'] + 'final_dataset.csv')
final_df = final_df.sort_values('date').reset_index(drop=True)

print(f'final_dataset shape: {final_df.shape}')
print(f'Columns: {list(final_df.columns)}')

# Compute daily log-returns
returns = final_df['Close'].pct_change().dropna().values
pol_company = final_df['polarity_company'].iloc[1:].values
pol_market  = final_df['polarity_market'].iloc[1:].values

print(f'returns shape      : {returns.shape}')
print(f'pol_company shape  : {pol_company.shape}')

granger_df = run_granger_tests(
    returns=returns,
    polarity_company=pol_company,
    polarity_market=pol_market,
    max_lag=CONFIG['granger_max_lag'],
)
print(granger_df.to_string())

granger_df.to_csv(CONFIG['tables_dir'] + 'granger_results.csv', index=False)
print(f"Saved -> {CONFIG['tables_dir']}granger_results.csv")

## Step 10 — Trading Simulation (Long-Only, Kotekar Algorithm 1)

In [ ]:
# ── Cell 12: Trading simulation ───────────────────────────────────────────
from src.evaluation.trading_sim import run_trading_simulation

# Actual next-day log returns for test period
# final_dataset rows correspond to test set (after split)
n_total = len(final_df)
test_start_idx = int(n_total * (CONFIG['train_ratio'] + CONFIG['val_ratio']))
test_df = final_df.iloc[test_start_idx:].reset_index(drop=True)

# Day-over-day returns for test set (aligned to predictions)
actual_returns = test_df['Close'].pct_change().fillna(0).values
# Trim to match prediction length (window creates window_size fewer samples)
actual_returns = actual_returns[CONFIG['window_size']:]

print(f'Test predictions shape: {y_clf_proba.shape}')
print(f'Actual returns shape  : {actual_returns.shape}')

sim_results = run_trading_simulation(
    y_pred_proba=y_clf_proba,
    actual_returns=actual_returns,
    risk_free_rate=CONFIG['risk_free_rate'],
    save_path=CONFIG['figures_dir'] + 'trading_simulation.png',
)

print('\nTrading simulation results:')
for k, v in sim_results.items():
    if isinstance(v, float):
        print(f'  {k:25s}: {v:.4f}')
    else:
        print(f'  {k:25s}: {v}')

In [ ]:
# ── Cell 13: Save trading results table ───────────────────────────────────
trading_row = {k: v for k, v in sim_results.items() if not hasattr(v, '__len__')}
pd.DataFrame([trading_row]).to_csv(
    CONFIG['tables_dir'] + 'trading_results.csv', index=False)
print(f"Saved -> {CONFIG['tables_dir']}trading_results.csv")

## Step 11 — Optional: Wavelet Denoising Visualisation

In [ ]:
# ── Cell 14: wavelet_denoising.png (optional) ─────────────────────────────
import pywt
from src.data.preprocessor import coif3_denoise

final_df2 = pd.read_csv(CONFIG['processed_data_dir'] + 'final_dataset.csv')
raw_close = final_df2['Close_d']  # denoised is in final_dataset
# We need the original close — load merged_data
merged = pd.read_csv(CONFIG['processed_data_dir'] + 'merged_data.csv')
merged = merged.dropna(subset=['Close']).reset_index(drop=True)

raw_prices = merged['Close'].values[:200]
denoised   = coif3_denoise(raw_prices, CONFIG)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(raw_prices, alpha=0.7, label='Raw Close', lw=1)
ax.plot(denoised,   alpha=0.9, label='Coif3 Denoised', lw=1.5)
ax.set_xlabel('Trading Day')
ax.set_ylabel('Nifty 50 Index')
ax.set_title('Coif3 Wavelet Denoising — Close Price (first 200 days)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CONFIG['figures_dir'] + 'wavelet_denoising.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved -> {CONFIG['figures_dir']}wavelet_denoising.png")

## Final Summary

In [ ]:
# ── Cell 15: Summary of all outputs ──────────────────────────────────────
figures = [
    'loss_curves.png', 'confusion_matrix.png', 'auc_roc_curve.png',
    'ablation_comparison.png', 'trading_simulation.png',
    'shap_summary.png', 'wavelet_denoising.png',
]
tables = [
    'ablation_summary.csv', 'granger_results.csv', 'trading_results.csv'
]

print('=' * 55)
print('NOTEBOOK 06 COMPLETE')
print('=' * 55)
print('FIGURES:')
for fn in figures:
    path = CONFIG['figures_dir'] + fn
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  {fn:35s}  {status}')

print('\nTABLES:')
for fn in tables:
    path = CONFIG['tables_dir'] + fn
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  {fn:35s}  {status}')

print(f'\nablation_results.csv: {CONFIG["ablation_results"]}')
df_check = pd.read_csv(CONFIG['ablation_results'])
print(df_check.groupby('config')['accuracy'].agg(['mean','max','std']).round(4).to_string())